In [1]:
import pandas as pd
import requests

canarias = pd.read_csv("../data/raw/canarias_pm10_samplingpoints.csv")
print(len(canarias), "puntos de medición")

API = "https://eeadmz1-downloads-api-appservice.azurewebsites.net"

body = {
    "countries": ["ES"],
    "cities": [],
    "pollutants": ["PM10"],
    "dataset": 2,
    "source": "API",
}

r = requests.post(f"{API}/ParquetFile/urls", json=body, timeout=120)
print(r.status_code)

60 puntos de medición
200


In [2]:
print(r.text[:1500])

ï»¿ParquetFileUrl
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_01055001_10_47.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_01059009_10_47.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_01059012_10_47.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_03009006_10_M.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_03014006_10_M.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_03031002_10_46.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_03065007_10_M.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_03133002_10_46.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_03140001_10_46.parquet
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p-e1a/ES/SP_04013004_10_49.parquet
https

In [3]:
from io import StringIO

texto = r.content.decode("utf-8-sig")          # bytes → texto, sin el BOM
urls = pd.read_csv(StringIO(texto))             # StringIO hace que un texto se lea como si fuera un archivo
urls["archivo"] = urls["ParquetFileUrl"].str.split("/").str[-1]   # solo el nombre del archivo

print(urls.shape)
urls.head()

(773, 2)


,ParquetFileUrl,archivo
0,https://eeadmz1batchservice02.blob.core.window...,SP_01055001_10_47.parquet
1,https://eeadmz1batchservice02.blob.core.window...,SP_01059009_10_47.parquet
2,https://eeadmz1batchservice02.blob.core.window...,SP_01059012_10_47.parquet
3,https://eeadmz1batchservice02.blob.core.window...,SP_03009006_10_M.parquet
4,https://eeadmz1batchservice02.blob.core.window...,SP_03014006_10_M.parquet


In [4]:
# 1. Sacar el código de 8 dígitos del nombre del archivo
urls["codigo"] = urls["archivo"].str.split("_").str[1]

# 2. Los códigos de tus estaciones de Canarias, como texto (para comparar texto con texto)
codigos_canarias = canarias["AirQualityStationNatCode"].astype(str)

# 3. Quedarse con las URLs cuyo código está en tu lista
urls_canarias = urls[urls["codigo"].isin(codigos_canarias)]

print(len(urls), "URLs de toda España")
print(len(urls_canarias), "URLs de Canarias")
urls_canarias.head()

773 URLs de toda España
105 URLs de Canarias


,ParquetFileUrl,archivo,codigo
251,https://eeadmz1batchservice02.blob.core.window...,SP_35002001_10_47.parquet,35002001
252,https://eeadmz1batchservice02.blob.core.window...,SP_35002001_10_A.parquet,35002001
254,https://eeadmz1batchservice02.blob.core.window...,SP_35004001_10_47.parquet,35004001
255,https://eeadmz1batchservice02.blob.core.window...,SP_35004001_10_A.parquet,35004001
256,https://eeadmz1batchservice02.blob.core.window...,SP_35004002_10_49.parquet,35004002


In [5]:
print(urls_canarias["codigo"].nunique(), "estaciones distintas")

urls_canarias["sufijo"] = urls_canarias["archivo"].str.split("_").str[3].str.replace(".parquet", "")
urls_canarias["sufijo"].value_counts()

47 estaciones distintas


sufijo
A     43
49    28
47    24
46    10
Name: count, dtype: int64

In [6]:
faltan = set(codigos_canarias) - set(urls_canarias["codigo"])
canarias[canarias["AirQualityStationNatCode"].astype(str).isin(faltan)][
    ["AirQualityStation", "AirQualityStationNatCode", "AirQualityStationType"]
].drop_duplicates("AirQualityStation")

,AirQualityStation,AirQualityStationNatCode,AirQualityStationType
0,STA-ES0197A,35026002,industrial
5,STA-ES1132A,38038011,industrial
7,STA-ES1133A,38011003,industrial
15,STA-ES1748A,35022001,industrial
17,STA-ES1757A,38038015,traffic


In [7]:
urls_por_prefijo = urls[urls["codigo"].str.startswith(("35", "38"))]
print(len(urls_por_prefijo), "URLs de Canarias filtrando directo")
print(urls_por_prefijo["codigo"].nunique(), "estaciones")

121 URLs de Canarias filtrando directo
56 estaciones


In [8]:
from pathlib import Path

destino = Path("../data/raw/eea_pm10")
destino.mkdir(parents=True, exist_ok=True)

def bajar(url):
    archivo = destino / url.split("/")[-1]   # mismo nombre que en la URL
    if not archivo.exists():                  # si ya lo tengo, no lo bajo de nuevo
        r = requests.get(url, timeout=120)
        r.raise_for_status()                  # si falla (404, 500...), que dé error y no guarde basura
        archivo.write_bytes(r.content)
    return archivo

prueba = urls_por_prefijo[urls_por_prefijo["codigo"] == "35002001"]
archivos = [bajar(u) for u in prueba["ParquetFileUrl"]]
archivos

[PosixPath('../data/raw/eea_pm10/SP_35002001_10_47.parquet'),
 PosixPath('../data/raw/eea_pm10/SP_35002001_10_A.parquet')]

In [9]:
for a in archivos:
    df = pd.read_parquet(a)
    print(a.name, "|", len(df), "filas |", df["Start"].min(), "→", df["Start"].max())

df.head()

SP_35002001_10_47.parquet | 34286 filas | 2013-01-01 01:00:00 → 2017-01-01 00:00:00
SP_35002001_10_A.parquet | 78144 filas | 2017-01-01 01:00:00 → 2026-01-01 00:00:00


,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,DataCapture,FkObservationLog
0,ES/SP_35002001_10_A,5,2017-01-01 01:00:00,2017-01-01 02:00:00,41.000000000000000000,ug.m-3,hour,1,1,2018-07-19 17:29:00,None,95656548-ac98-45c4-bb32-880b0fecb508
1,ES/SP_35002001_10_A,5,2017-01-01 02:00:00,2017-01-01 03:00:00,22.000000000000000000,ug.m-3,hour,1,1,2018-07-19 17:29:00,None,95656548-ac98-45c4-bb32-880b0fecb508
2,ES/SP_35002001_10_A,5,2017-01-01 03:00:00,2017-01-01 04:00:00,35.000000000000000000,ug.m-3,hour,1,1,2018-07-19 17:29:00,None,95656548-ac98-45c4-bb32-880b0fecb508
3,ES/SP_35002001_10_A,5,2017-01-01 04:00:00,2017-01-01 05:00:00,34.000000000000000000,ug.m-3,hour,1,1,2018-07-19 17:29:00,None,95656548-ac98-45c4-bb32-880b0fecb508
4,ES/SP_35002001_10_A,5,2017-01-01 05:00:00,2017-01-01 06:00:00,32.000000000000000000,ug.m-3,hour,1,1,2018-07-19 17:29:00,None,95656548-ac98-45c4-bb32-880b0fecb508
